# P3 & P6 — chọn-epoch-tốt-nhất · MIMIC 3/6/10% + IU 3% + ablation

Chạy P3/P6 với `eval_test_every_epoch=1` → mỗi epoch ghi metrics trên **D_t_final** vào
`test_history_*.csv`. Cell cuối tự **dò epoch gần GOLD nhất** (như cách tái lập Forget-MI).

**Cell 2**: chọn `DATASET` (mimic/iu), `FORGET_PCT`, phương pháp, ablation.
Mỗi run lâu hơn (~1.5–2h) vì eval mỗi epoch — đáng, để có checkpoint tốt nhất.


In [ ]:
# Cell 1: setup
import os, subprocess
WORK='/kaggle/working'; REPO=f'{WORK}/Forget-MI-LoKU'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/nhnhu146/Forget-MI-LoKU.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
os.chdir(REPO)
assert os.path.exists('training/adv_common.py'),'push code truoc + re-import notebook'
subprocess.run(['pip','install','-q','pydicom','scikit-image','scikit-learn','pyyaml','wandb','seaborn==0.13.2'],check=True)
subprocess.run(['pip','install','-q','transformers==4.38.0','peft==0.10.0','accelerate==0.27.0'],check=True)
import torch; assert torch.cuda.is_available(),'Bat GPU'
print('Commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('GPU   :',torch.cuda.get_device_name(0))


In [ ]:
# Cell 2: CO CHON + path discovery (mimic / iu)
import glob, os

DATASET    = 'mimic'   # 'mimic' | 'iu'
FORGET_PCT = 3         # 3 | 6 | 10   (iu: chi 3)
SEED       = 42

RUN_P3 = True
RUN_P6 = True

RUN_EVAL_REF = True    # eval OG + GOLD tren D_t_final (moc vang)

# eval MOI epoch de CHON EPOCH TOT NHAT (gan GOLD nhat) - GIONG HET cach da chon Forget-MI.
# BAT (True) de P3/P6 nhat quan voi baseline -> so sanh cong bang. ~1.5-2h/run (co 4 account nen ok).
# (Ky vong: P3/P6 khong over-forget nen epoch tot nhat thuong ~E30 - eval de XAC NHAN + cong bang.)
EVAL_EVERY_EPOCH = True

# Ablation (chi chay khi RUN_ABLATIONS=True). Moi cai chay tren CA P3 lan P6 (tru p6_gate_*).
RUN_ABLATIONS = False
ABLATIONS = ['no_uumu','no_fila','no_noise']   # + 'p6_gate_free','p6_gate_reg' (chi P6)

assert DATASET in ('mimic','iu')
assert FORGET_PCT in (3,6,10)
if DATASET=='iu': assert FORGET_PCT==3, 'IU chi co 3%'

def fd(*slugs):
    for s in slugs:
        if os.path.isdir(f'/kaggle/input/{s}'): return f'/kaggle/input/{s}'
        h=glob.glob(f'/kaggle/input/datasets/*/{s}')
        if h: return sorted(h)[0]
    return None
def bins(root):
    return sorted(glob.glob(os.path.join(root,'**','pytorch_model.bin'),recursive=True),key=len)

if DATASET=='mimic':
    DATA=fd('forget-mi-data'); MOD=fd('forget-mi-models-full','forget-mi-models')
    assert DATA and MOD,'Add forget-mi-data + forget-mi-models-full'
    BASE=os.path.dirname(bins(os.path.join(MOD))[0]) if False else \
         os.path.dirname([b for b in bins(MOD) if 'training_original_model' in b][0])
    gold=[b for b in bins(MOD) if f'model_retrained_{FORGET_PCT}per' in b]
    GOLD=os.path.dirname(gold[0]) if gold else BASE
    TEXT=os.path.join(DATA,'data','metadata'); IMG=os.path.join(DATA,'data','img_data')
    SPLIT='./data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv'
    FORGET=f'./data_splits/forget_set_{FORGET_PCT}per.csv'
    tag=f'{FORGET_PCT}per'
else:  # IU
    DATA=fd('forget-mi-data-iu'); MOD=fd('forget-mi-models-iu'); MODRE=fd('forget-mi-models-iu-re')
    RAD=fd('chest-xrays-indiana-university')
    assert DATA and MOD and RAD,'Add forget-mi-data-iu + forget-mi-models-iu + chest-xrays-indiana-university'
    ogb=[b for b in bins(MOD) if 'model_og' in b.lower() or 'base_model' in b.lower()] or bins(MOD)
    BASE=os.path.dirname(ogb[0])
    reb=(bins(MODRE) if MODRE else []) or [b for b in bins(MOD) if 'retrain' in b.lower()]
    GOLD=os.path.dirname(reb[0]) if reb else BASE
    TEXT=os.path.join(DATA,'data','metadata')
    IMG=fd('chest-xrays-indiana-university')  # anh IU o raddar
    # split + forget nam TRONG forget-mi-data-iu
    sp=glob.glob(os.path.join(DATA,'**','iu-split.csv'),recursive=True) or \
       glob.glob(os.path.join(DATA,'**','*iu*split*.csv'),recursive=True)
    SPLIT=sp[0] if sp else None
    fg=glob.glob(os.path.join(DATA,'**',f'forget_set_{FORGET_PCT}per_iu.csv'),recursive=True)
    FORGET=fg[0] if fg else None
    assert SPLIT and FORGET, f'Khong thay iu-split / forget_set_iu trong {DATA}'
    tag=f'iu{FORGET_PCT}per'

for n,p in {'BASE':BASE,'TEXT':TEXT,'IMG':IMG,'SPLIT':SPLIT,'FORGET':FORGET}.items():
    assert p and os.path.exists(p),f'Missing {n}: {p}'

OUT=f'/kaggle/working/adv_{tag}_s{SEED}'
RESULTS=f'/kaggle/working/results_{tag}.csv'          # eval cuoi (last + selected + gold-best)
def hist(rid): return f'/kaggle/working/perepoch_{rid}.csv'          # S_val trajectory
def thist(rid): return f'/kaggle/working/test_history_{rid}.csv'     # D_t_final trajectory (chon epoch)

COMMON={'forget_set_path':FORGET,'base_model_path':BASE,'bert_pretrained_dir':BASE,
        'retrained_model_path':GOLD,'text_data_dir':TEXT,'img_data_dir':IMG,
        'data_split_path':SPLIT,'output_dir':OUT,'results_csv_path':RESULTS,
        'eval_test_every_epoch':(1 if EVAL_EVERY_EPOCH else 0)}
print('DATASET',DATASET,'| PCT',FORGET_PCT,'| SEED',SEED,'| tag',tag)
print('BASE',BASE); print('GOLD',GOLD); print('SPLIT',SPLIT); print('FORGET',FORGET)


In [ ]:
# Cell 3: chay P3 / P6 (moi run: chon-epoch tren D_t_final)
import os, subprocess, time
LOG=[]
def run(script, rid, extra=None):
    ovr=dict(COMMON); ovr['id']=rid; ovr['history_csv_path']=hist(rid); ovr['test_history_csv_path']=thist(rid)
    if extra: ovr.update(extra)
    arg=','.join(f'{k}={v}' for k,v in ovr.items())
    env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled','PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}
    cmd=['python',script,'--config','config_advanced_kaggle.yaml','--seed',str(SEED),'--fresh','--override',arg]
    print('='*70+f'\nRUN {rid}\n'+'='*70); t0=time.time()
    try:
        subprocess.run(cmd,env=env,check=True); LOG.append((rid,'OK',round((time.time()-t0)/3600,2)))
    except subprocess.CalledProcessError as e:
        print(f'FAIL {rid} rc={e.returncode}'); LOG.append((rid,f'FAIL{e.returncode}',round((time.time()-t0)/3600,2)))

if RUN_P3: run('training/forgetmi_p3.py', f'p3_{tag}_s{SEED}')
if RUN_P6: run('training/forgetmi_p6.py', f'p6_{tag}_s{SEED}')
print('\nTONG KET:'); [print(' ',*x) for x in LOG]


In [ ]:
# Cell 4: eval OG + GOLD tren D_t_final (moc vang) — chi mimic (iu: gold la model_retrained_iu)
import os, subprocess
def evalref(label, mpath, mtype='pretrained'):
    ovr=dict(COMMON); ovr['results_csv_path']=RESULTS
    arg=','.join(f'{k}={v}' for k,v in ovr.items())
    env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled'}
    cmd=['python','training/forgetmi_eval_only.py','--config','config_advanced_kaggle.yaml','--seed',str(SEED),
         '--label',label,'--model_type',mtype,'--model_path',mpath,'--method','reference','--override',arg]
    print('eval-ref',label)
    try: subprocess.run(cmd,env=env,check=True)
    except subprocess.CalledProcessError as e: print('FAIL',label,e.returncode)
if RUN_EVAL_REF:
    evalref(f'og_{tag}', BASE); evalref(f're_{tag}', GOLD)
else:
    print('RUN_EVAL_REF=False -> bo qua')


In [ ]:
# Cell 5: ABLATION (bat RUN_ABLATIONS). Moi ablation = 1 override.
import os, subprocess, time
ABL={
 'no_uumu':   {'ablate_uu_mu':1},          # tat quen bieu dien (UU/MU) -> do dong gop
 'no_fila':   {'loku_random_init':1},      # bo Fisher/FILA init
 'no_noise':  {'use_noise':0},             # bo nhieu (og_rnd = anh sach)
 'p6_gate_free':{'gate_mode':'free'},      # gate trainable tu do (chi P6)
 'p6_gate_reg': {'gate_mode':'reg'},       # gate + phat lech (chi P6)
}
def runa(script, rid, extra):
    ovr=dict(COMMON); ovr['id']=rid; ovr['history_csv_path']=hist(rid); ovr['test_history_csv_path']=thist(rid); ovr.update(extra)
    arg=','.join(f'{k}={v}' for k,v in ovr.items())
    env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled','PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}
    cmd=['python',script,'--config','config_advanced_kaggle.yaml','--seed',str(SEED),'--fresh','--override',arg]
    print('='*70+f'\nABLATION {rid}\n'+'='*70); t0=time.time()
    try: subprocess.run(cmd,env=env,check=True); LOG.append((rid,'OK',round((time.time()-t0)/3600,2)))
    except subprocess.CalledProcessError as e: print('FAIL',rid,e.returncode); LOG.append((rid,f'FAIL{e.returncode}',0))
if RUN_ABLATIONS:
    for a in ABLATIONS:
        if a not in ABL: print('bo qua (khong biet):',a); continue
        if a.startswith('p6_'):
            runa('training/forgetmi_p6.py', f'p6_{a}_{tag}_s{SEED}', ABL[a])
        else:
            if RUN_P3: runa('training/forgetmi_p3.py', f'p3_{a}_{tag}_s{SEED}', ABL[a])
            if RUN_P6: runa('training/forgetmi_p6.py', f'p6_{a}_{tag}_s{SEED}', ABL[a])
    print('\nTONG KET ABLATION:'); [print(' ',*x) for x in LOG]
else:
    print('RUN_ABLATIONS=False -> bo qua')


In [ ]:
# Cell 6: BANG KET QUA
#  - EVAL_EVERY_EPOCH=False (mac dinh): in bang tu results_*.csv (last + selected)
#  - EVAL_EVERY_EPOCH=True: them phan DO EPOCH TOT NHAT tu test_history_*.csv
import os, glob, pandas as pd
pd.set_option('display.width',220); pd.set_option('display.max_columns',40)

if os.path.exists(RESULTS):
    df=pd.read_csv(RESULTS)
    c=[x for x in ['method','checkpoint_kind','id','Forget_AUC','Forget_Macro_F1','Test_AUC',
       'Test_Macro_F1','MIA','MIA_paper','forget_ce','test_ce','unlearn_core_hours'] if x in df.columns]
    print('===== KET QUA (results) ====='); print(df[c].to_string(index=False))
else:
    print('chua co',RESULTS)

if not glob.glob('/kaggle/working/test_history_*.csv'):
    print('\n(EVAL_EVERY_EPOCH=False -> dung `last` E30 lam ket qua chinh. Xong.)')
    print('TAI VE: results_*.csv'); raise SystemExit
print('\n===== DO EPOCH TOT NHAT (EVAL_EVERY_EPOCH=True) =====')

# moc vang tu Cell 4 (results_*.csv, method=reference)
gold=None
if os.path.exists(RESULTS):
    dr=pd.read_csv(RESULTS)
    g=dr[(dr.get('method')=='reference') & dr['run_id'].astype(str).str.startswith('re')]
    if len(g): gold=g.iloc[-1]
if gold is not None:
    gDf,gMIA,gDt=float(gold['Forget_AUC']),float(gold['MIA']),float(gold['Test_AUC'])
    print(f'GOLD: Df-AUC {gDf:.3f}  MIA {gMIA:.3f}  Dt-AUC {gDt:.3f}\n')
else:
    gDf,gMIA,gDt=0.5,0.42,0.62; print('CHUA co GOLD -> dung moc mac dinh 0.5/0.42/0.62\n')

rows=[]
for f in sorted(glob.glob('/kaggle/working/test_history_*.csv')):
    pe=pd.read_csv(f); rid=os.path.basename(f)[len('test_history_'):-4]
    # score = |Df-gold| + |MIA-gold| + phat neu Dt tut DUOI gold (Dt cao hon gold = tot, khong phat)
    pe['score']=(pe['Df_AUC']-gDf).abs()+(pe['MIA']-gMIA).abs()+(gDt-pe['Dt_AUC']).clip(lower=0)
    b=pe.loc[pe['score'].idxmin()]
    print(f'=== {rid} ===')
    print(pe[['epoch','Df_AUC','Df_F1','Dt_AUC','Dt_F1','MIA','MIA_paper','forget_ce','test_ce']].to_string(index=False))
    print(f'>>> EPOCH TOT NHAT = E{int(b.epoch)}  Df-AUC {b.Df_AUC:.3f} Df-F1 {b.Df_F1:.3f} '
          f'Dt-AUC {b.Dt_AUC:.3f} Dt-F1 {b.Dt_F1:.3f} MIA {b.MIA:.3f}/{b.MIA_paper:.3f}\n')
    rows.append({'run':rid,'best_epoch':int(b.epoch),'Df_AUC':b.Df_AUC,'Df_F1':b.Df_F1,
                 'Dt_AUC':b.Dt_AUC,'Dt_F1':b.Dt_F1,'MIA':b.MIA,'MIA_paper':b.MIA_paper,
                 'forget_ce':b.forget_ce,'test_ce':b.test_ce})
if rows:
    out=pd.DataFrame(rows); out.to_csv('/kaggle/working/best_epoch_summary.csv',index=False)
    print('===== BANG best-epoch (moi run) ====='); print(out.to_string(index=False))
print('\nTAI VE: results_*.csv + test_history_*.csv + perepoch_*.csv + best_epoch_summary.csv')
